In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce

# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import sys
from pathlib import Path

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from chronopack.preprocessing import (
    TIME_GROUP_CATEGORIES,
    add_time_metadata,
    annotate_qc_flags,
    filter_cells_by_qc,
    filter_technical_genes,
    load_time_tagged_anndata,
    map_time_to_group,
)
from chronopack.trajectories import (
    CalculateTerminalFatesAcrossCells,
    PlotHardClusterMassProportionsOverTime,
    cosine_topk_leiden_from_terminal_dist,
    extract_wot_couplings,
    fate_umap_2x2_scanpy,
    mass_retention_plots_and_umap,
    run_wot_transport_workflow,
    sparse_adjacency_from_tmats,
    terminal_fates_from_terminal_clusters,
    topk_csr_rows,
)


In [ ]:
sample_paths = [
    '../input/N701_scK562_TNFA_Puromycin_RT2.aligned.sorted.merged.tagged.dge.time_tags.csv.gz',
    '../input/N702_scK562_TNFA_Puromycin_RT3.aligned.sorted.merged.tagged.dge.time_tags.csv.gz',
    '../input/N704_scK562_TNFA_Puromycin_RT4.aligned.sorted.merged.tagged.dge.time_tags.csv.gz',
]

adata = load_time_tagged_anndata(sample_paths, sample_names=['N701', 'N702', 'N704'])
print(adata)


In [ ]:
cell_cycle_genes = [x.strip() for x in open('/new-stg/home/alex/ref/gene_sets/regev_lab_cell_cycle_genes.txt')]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]
cell_cycle_genes = [x for x in cell_cycle_genes if x in adata.var_names]

In [ ]:
annotate_qc_flags(adata)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True)


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12,4))
sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.4, ax=axs[0], show=False)
sc.pl.violin(adata, 'total_counts',      jitter=0.4, ax=axs[1], show=False)
sc.pl.violin(adata, 'pct_counts_mt',     jitter=0.4, ax=axs[2], show=False)

axs[0].set_ylim(0, 2000)    
axs[1].set_ylim(0, 5000)    
axs[2].set_ylim(0, 10)      

plt.show()

In [ ]:
adata = filter_cells_by_qc(adata, min_counts=200, max_counts=10000, max_pct_mt=7.5, min_genes=300)
sc.pp.filter_genes(adata, min_cells=3)
print(adata)


In [ ]:
sc.pp.scrublet(adata, batch_key="sample")
adata = adata[~adata.obs['predicted_doublet'], :]

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e6)
adata.layers['cpm'] = adata.X.copy()
sc.pp.log1p(adata)
adata.layers['log1p'] = adata.X.copy()

sc.pp.highly_variable_genes(adata, batch_key='sample', n_top_genes=3000)
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
adata_full = adata.copy()
adata = adata[:, adata.var.highly_variable]

In [ ]:
bck = adata.copy()

In [ ]:
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt', 'S_score', 'G2M_score'])
sc.pp.scale(adata)
sc.tl.pca(adata, n_comps=50, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, n_pcs=30, log=True)
sc.pp.neighbors(adata, n_pcs=12)

sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.1, key_added='leiden_unintegrated')

sc.pl.pca_scatter(adata, color='phase')
sc.pl.umap(adata, color=['sample', 'phase', 'NFKBIA', 'ICAM1', 'CXCL3', 'time_group', 'total_counts', 'doublet_score', 'pct_counts_mt', 'pct_counts_rb'], wspace=0.4, size=40, ncols=4)

In [ ]:
sc.tl.leiden(adata, resolution=0.5)
sc.pl.umap(adata, color=['leiden'], wspace=0.4, size=40, ncols=3)

In [ ]:
adata = adata[adata.obs['leiden'] != '4', :]

sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.1, key_added='leiden_unintegrated')
sc.pl.umap(adata, color=['leiden'], wspace=0.4, size=40, ncols=3)

In [ ]:
sc.pl.umap(adata, color=['sample', 'phase', 'total_counts', 'doublet_score', 'pct_counts_mt', 'pct_counts_rb', 'pct_counts_mt', 'n_genes', 'n_genes_by_counts'], wspace=0.4, size=40, ncols=4)

In [ ]:
bck = bck[adata.obs.index, :]

In [ ]:
adata_c = bck.copy()

# Batch correction and residualization used by Waddington-OT downstream.
sc.pp.combat(adata_c, key="sample")
sc.pp.regress_out(adata_c, keys=["total_counts", "pct_counts_mt", "S_score", "G2M_score"])
adata_c.layers["corrected"] = adata_c.X.copy()

# Diagnostic embedding for filtering and QC review.
sc.pp.scale(adata_c, max_value=10)
sc.tl.pca(adata_c, n_comps=50)
sc.pp.neighbors(adata_c, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_c)
sc.tl.leiden(adata_c)

sc.pl.umap(
    adata_c,
    color=["sample", "phase", "NFKBIA", "ICAM1", "CXCL3", "time_group", "total_counts", "n_genes_by_counts"],
    wspace=0.4,
    size=40,
    ncols=3,
)


In [ ]:
sc.pl.umap(adata_c, color=['sample', 'phase', 'total_counts', 'doublet_score', 'pct_counts_mt', 'pct_counts_rb', 'pct_counts_mt', 'n_genes', 'n_genes_by_counts'], wspace=0.4, size=40, ncols=4)

In [ ]:
sc.tl.leiden(adata_c, resolution=0.5)
sc.pl.umap(adata_c, color=['leiden'], wspace=0.4, size=40, ncols=3)

In [ ]:
adata_c = adata_c[adata_c.obs['leiden'] != '3', :].copy()


In [ ]:
bck = bck[adata_c.obs.index, :]

# Save preprocessing checkpoint


In [ ]:
adata_hvg = bck.copy()
adata_hvg.layers["corrected"] = adata_c[adata_hvg.obs_names, :].layers["corrected"].copy()

adata_full_chk0 = adata_full[adata_hvg.obs_names, :].copy()
adata_full_chk0.obs = adata_hvg.obs.copy()
adata_full_chk0.write_h5ad("../output/adata_full.chk0.h5ad")
adata_hvg.write_h5ad("../output/adata_hvg.chk0.h5ad")
